# Airline Satisfaction — Experiment 1: pretrained TabICLv2

Direct inference using frozen TabICLv2 classifier weights from `jingang/TabICL`, compared with a locally trained LightGBM control. No public submission files or target labels are reused.

Attach **Predicting Airline Satisfaction (playground-series-s6e10)**, enable Internet for the first checkpoint download, and run cells in order. This CPU recipe matches the local experiment and may take tens of minutes. Outputs include validation metrics, separate model predictions, and `submission_selected.csv`. It does not submit automatically.

Validation: 599,635 rows form the base-model training pool, 50,000 select among five fixed blend weights, and another 50,000 report the selected recipe. LightGBM early stopping uses a further split inside the training pool. TabICL uses 16,384 context examples; it is not trained from scratch or fine-tuned. This is a single holdout experiment, not cross-validation.

Sources and attribution:
- TabICLv2 official implementation: https://github.com/soda-inria/tabicl
- Official pretrained checkpoint (BSD-3-Clause): https://huggingface.co/jingang/TabICL
- Competition: https://www.kaggle.com/competitions/playground-series-s6e10
- Qu et al. (2026), TabICLv2: https://arxiv.org/abs/2602.11139


In [ ]:
%pip install -q tabicl==2.2.0 lightgbm


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
checkpoint = hf_hub_download(repo_id='jingang/TabICL', filename='tabicl-classifier-v2-20260212.ckpt')
paths=[p.parent for p in Path('/kaggle/input').rglob('train.csv') if 'playground-series-s6e10' in str(p) and (p.parent/'test.csv').exists()]
if len(paths)!=1: raise ValueError(f'Attach exactly one copy of the competition data: {paths}')
DATA_DIR=paths[0]


In [ ]:
source = '"""Experiment 1: direct pretrained TabICLv2 inference and a LightGBM control.\n\nThe transformer weights stay frozen. Only labeled context/preprocessing is fitted.\nSelection and reporting partitions are disjoint; neither trains base models.\n"""\nfrom pathlib import Path\nimport argparse, hashlib, importlib.metadata, json, time, os, subprocess, sys\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score, log_loss, brier_score_loss\nimport joblib\n\nCHECKPOINT = \'tabicl-classifier-v2-20260212.ckpt\'\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef scores(y,p):\n    return dict(auc=float(roc_auc_score(y,p)),log_loss=float(log_loss(y,p)),\n                brier=float(brier_score_loss(y,p)))\n\ndef lgb_features(x, levels=None):\n    x=x.copy()\n    if levels is None:\n        levels={c:sorted(x[c].dropna().unique().tolist()) for c in x.select_dtypes(include=[\'object\',\'string\'])}\n    for c,lv in levels.items(): x[c]=pd.Categorical(x[c],categories=lv)\n    return x,levels\n\ndef predict_batches(model,x,path,batch=2048):\n    """Resume only within a fingerprint-specific run directory."""\n    path.mkdir(parents=True,exist_ok=True)\n    out=[]\n    positive=list(model.classes_).index(1)\n    for start in range(0,len(x),batch):\n        f=path/f\'{start:07d}.npy\'; end=min(start+batch,len(x))\n        if f.exists(): p=np.load(f,allow_pickle=False)\n        else:\n            p=model.predict_proba(x.iloc[start:end])[:,positive]\n            np.save(f,p)\n        if p.shape!=(end-start,) or not np.isfinite(p).all(): raise ValueError(f\'Invalid batch {f}\')\n        out.append(p)\n        if start//batch%20==0 or end==len(x): print(f\'TabICL {path.name}: {end}/{len(x)}\',flush=True)\n    return np.concatenate(out)\n\ndef transformer_worker(config_file):\n    from tabicl import TabICLClassifier\n    import torch\n    torch.set_num_threads(4)\n    torch.set_num_interop_threads(1)\n    c=json.loads(Path(config_file).read_text())\n    train=pd.read_csv(c[\'train\']); y=train.satisfaction.astype(int).to_numpy()\n    x=train.drop(columns=[\'id\',\'satisfaction\']);ctx=np.load(c[\'context\'])\n    if c[\'phase\']==\'held\': xp=x.iloc[np.load(c[\'rows\'])]\n    else: xp=pd.read_csv(c[\'test\']).drop(columns=[\'id\'])\n    model=TabICLClassifier(n_estimators=c[\'estimators\'],n_jobs=4,device=\'cpu\',kv_cache=True,batch_size=1,\n        model_path=c[\'checkpoint\'],allow_auto_download=False,random_state=42)\n    print(\'Building pretrained context\',c[\'phase\'],len(ctx),flush=True)\n    model.fit(x.iloc[ctx],y[ctx]);print(\'Pretrained context ready\',c[\'phase\'],flush=True)\n    pred=predict_batches(model,xp,Path(c[\'batches\']))\n    np.save(c[\'output\'],pred)\n\n\ndef transformer_process(data_dir,checkpoint,ctx,rows,cache,estimators,phase):\n    ctxfile=(cache/f\'{phase}_context.npy\').resolve();np.save(ctxfile,ctx)\n    rowfile=(cache/f\'{phase}_rows.npy\').resolve();np.save(rowfile,rows)\n    outfile=(cache/f\'{phase}_pred.npy\').resolve()\n    if not outfile.exists():\n        c=dict(train=str((data_dir/\'train.csv\').resolve()),test=str((data_dir/\'test.csv\').resolve()),\n               context=str(ctxfile),rows=str(rowfile),phase=phase,checkpoint=str(checkpoint.resolve()),\n               estimators=estimators,batches=str((cache/phase).resolve()),output=str(outfile))\n        configfile=(cache/f\'{phase}_worker.json\').resolve();configfile.write_text(json.dumps(c))\n        env=os.environ.copy();env.update(OMP_NUM_THREADS=\'4\',MKL_NUM_THREADS=\'4\',OPENBLAS_NUM_THREADS=\'4\')\n        subprocess.run([sys.executable,str(Path(__file__).resolve()),\'--worker-config\',str(configfile)],check=True,env=env)\n    return np.load(outfile,allow_pickle=False)\n\n\ndef run(data_dir,output_dir,checkpoint,context=16384,estimators=2):\n    import lightgbm as lgb\n    t0=time.time(); data_dir=Path(data_dir); output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    checkpoint=Path(checkpoint)\n    train=pd.read_csv(data_dir/\'train.csv\');test=pd.read_csv(data_dir/\'test.csv\');sample=pd.read_csv(data_dir/\'sample_submission.csv\')\n    assert train.id.is_unique and test.id.is_unique and sample.id.is_unique\n    assert np.array_equal(test.id,sample.id) and not set(train.id)&set(test.id)\n    assert set(train.satisfaction.unique())=={False,True}\n    y=train.satisfaction.astype(int).to_numpy();x=train.drop(columns=[\'id\',\'satisfaction\']);xt=test.drop(columns=[\'id\'])\n    assert list(x)==list(xt)\n    packages={p:importlib.metadata.version(p) for p in [\'tabicl\',\'torch\',\'lightgbm\',\'scikit-learn\',\'pandas\',\'numpy\']}\n    config=dict(seed=42,context=context,estimators=estimators,checkpoint_sha256=sha(checkpoint),\n                train_sha256=sha(data_dir/\'train.csv\'),test_sha256=sha(data_dir/\'test.csv\'),\n                source_sha256=sha(__file__),packages=packages)\n    fingerprint=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()\n    cache=output_dir/f\'cache_{fingerprint[:16]}\';cache.mkdir(exist_ok=True)\n    pool,held=train_test_split(np.arange(len(y)),test_size=100000,random_state=42,stratify=y)\n    selection,report=train_test_split(held,test_size=.5,random_state=2026,stratify=y[held])\n    inner,stop=train_test_split(pool,test_size=.15,random_state=314159,stratify=y[pool])\n    ctx,_=train_test_split(pool,train_size=context,random_state=77,stratify=y[pool])\n    assert not set(pool)&set(held) and not set(selection)&set(report)\n    pd.DataFrame({\'id\':train.id,\'partition\':np.select([np.isin(np.arange(len(y)),selection),np.isin(np.arange(len(y)),report)],[\'selection\',\'report\'],default=\'train_pool\')}).to_csv(output_dir/\'partitions.csv\',index=False)\n    audit=dict(train_shape=list(train.shape),test_shape=list(test.shape),positive_fraction=float(y.mean()),\n               missing_train=train.isna().sum().to_dict(),missing_test=test.isna().sum().to_dict(),\n               duplicate_feature_rows=int(x.duplicated().sum()),features=list(x),config=config,fingerprint=fingerprint,\n               n_train_pool=len(pool),n_selection=len(selection),n_report=len(report))\n    (output_dir/\'audit.json\').write_text(json.dumps(audit,indent=2))\n    print(\'Data audit\',train.shape,test.shape,\'positive\',y.mean(),flush=True)\n    pfile=cache/\'validation.npz\'\n    params=dict(objective=\'binary\',learning_rate=.04,num_leaves=31,min_child_samples=100,\n                colsample_bytree=.9,subsample=.8,subsample_freq=1,reg_lambda=5,\n                n_jobs=4,random_state=42,verbosity=-1,deterministic=True,force_col_wise=True)\n    if pfile.exists():\n        d=np.load(pfile); tab=d[\'tab\'];gb=d[\'gb\'];best=int(d[\'best\'])\n    else:\n        a,levels=lgb_features(x.iloc[inner]);b,_=lgb_features(x.iloc[stop],levels)\n        early=lgb.LGBMClassifier(**params,n_estimators=2000)\n        early.fit(a,y[inner],eval_set=[(b,y[stop])],eval_metric=\'auc\',callbacks=[lgb.early_stopping(100,verbose=False)])\n        best=int(early.best_iteration_);print(\'LightGBM iterations\',best,flush=True)\n        a,levels=lgb_features(x.iloc[pool]);b,_=lgb_features(x.iloc[held],levels)\n        control=lgb.LGBMClassifier(**params,n_estimators=best).fit(a,y[pool]);gb=control.predict_proba(b)[:,1]\n        print(\'LightGBM validation predictions complete\',flush=True)\n        tab=transformer_process(data_dir,checkpoint,ctx,held,cache,estimators,\'held\')\n        np.savez_compressed(pfile,tab=tab,gb=gb,best=best)\n    selmask=np.isin(held,selection);repmask=np.isin(held,report)\n    weights=[0.,.25,.5,.75,1.]\n    candidates=[dict(tabicl_weight=w,**scores(y[held][selmask],(w*tab+(1-w)*gb)[selmask])) for w in weights]\n    weight=max(candidates,key=lambda d:d[\'auc\'])[\'tabicl_weight\']\n    metrics=dict(config=config,fingerprint=fingerprint,context_rows=context,best_lgb_iteration=best,\n                 selection_candidates=candidates,selected_tabicl_weight=weight,\n                 report={n:scores(y[held][repmask],p[repmask]) for n,p in {\'tabicl\':tab,\'lightgbm\':gb,\'selected\':weight*tab+(1-weight)*gb}.items()},\n                 notes=[\'Report partition never selects weights, context, preprocessing, or stopping iteration.\',\n                        \'One fixed stratified holdout, not cross-validation. Public test score is separate.\',\n                        \'TabICLv2 frozen pretrained weights; 16384 labeled context rows by default; no gradient fine-tuning.\',\n                        \'Final deployment refits context and LightGBM using all available labeled rows.\'])\n    (output_dir/\'metrics.json\').write_text(json.dumps(metrics,indent=2));print(json.dumps(metrics[\'report\'],indent=2),flush=True)\n    pd.DataFrame({\'id\':train.id.iloc[held].to_numpy(),\'y\':y[held],\'partition\':np.where(selmask,\'selection\',\'report\'),\'tabicl\':tab,\'lightgbm\':gb}).to_csv(output_dir/\'validation_predictions.csv\',index=False)\n    final=cache/\'test.npz\'\n    if final.exists():\n        d=np.load(final);pt=d[\'tab\'];pg=d[\'gb\']\n    else:\n        allctx,_=train_test_split(np.arange(len(y)),train_size=context,random_state=77,stratify=y)\n        pt=transformer_process(data_dir,checkpoint,allctx,np.arange(len(test)),cache,estimators,\'test\')\n        np.save(output_dir/\'tabicl_context_ids.npy\',train.id.iloc[allctx].to_numpy())\n        a,levels=lgb_features(x);b,_=lgb_features(xt,levels)\n        control=lgb.LGBMClassifier(**params,n_estimators=best).fit(a,y);pg=control.predict_proba(b)[:,1]\n        joblib.dump({\'model\':control,\'categories\':levels,\'features\':list(x)},output_dir/\'lightgbm.joblib\')\n        np.savez_compressed(final,tab=pt,gb=pg)\n    for name,p in {\'tabicl\':pt,\'lightgbm\':pg,\'selected\':weight*pt+(1-weight)*pg}.items():\n        assert p.shape==(len(test),) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()\n        out=sample.copy();out.satisfaction=p;out.to_csv(output_dir/f\'submission_{name}.csv\',index=False)\n    metrics[\'runtime_seconds\']=time.time()-t0;metrics[\'submission_sha256\']=sha(output_dir/\'submission_selected.csv\')\n    (output_dir/\'metrics.json\').write_text(json.dumps(metrics,indent=2));print(\'Experiment completed\',metrics[\'runtime_seconds\'],flush=True)\n    return metrics\n\nif __name__==\'__main__\':\n    if len(sys.argv)>1 and sys.argv[1]==\'--worker-config\':\n        transformer_worker(sys.argv[2]);sys.exit(0)\n    p=argparse.ArgumentParser();p.add_argument(\'--data-dir\',default=\'airline_satisfaction/data\');p.add_argument(\'--output-dir\',default=\'airline_satisfaction/experiments/v1/results\');p.add_argument(\'--checkpoint\',default=f\'airline_satisfaction/models/{CHECKPOINT}\');p.add_argument(\'--context\',type=int,default=16384);p.add_argument(\'--estimators\',type=int,default=2)\n    args=p.parse_args();run(args.data_dir,args.output_dir,args.checkpoint,args.context,args.estimators)\n'
Path("run_experiment.py").write_text(source)


In [ ]:
from run_experiment import run
metrics=run(DATA_DIR,Path('/kaggle/working/experiment1'),checkpoint)
metrics['report']


In [ ]:
import pandas as pd
from IPython.display import display, FileLink
submission_path=Path('/kaggle/working/experiment1/submission_selected.csv')
display(pd.read_csv(submission_path).head())
display(FileLink(str(submission_path)))
